# Day 042 Solution — SQL Fundamentals

setup_db, run_query, filter_orders, group_revenue, join_summary. All data and functions defined inline. Uses in-memory SQLite.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sqlite3
import pandas as pd


import sqlite3

def setup_db(conn):
    cur = conn.cursor()
    cur.execute('''
        CREATE TABLE IF NOT EXISTS orders (
            order_id  INTEGER PRIMARY KEY,
            product   TEXT,
            category  TEXT,
            region    TEXT,
            price     REAL,
            quantity  INTEGER,
            revenue   REAL
        )''')
    cur.execute('''
        CREATE TABLE IF NOT EXISTS products (
            product    TEXT PRIMARY KEY,
            category   TEXT,
            unit_price REAL
        )''')
    rows = [
        (1,'Widget','Electronics','North',25.0,10,250.0),
        (2,'Gadget','Electronics','South',150.0,3,450.0),
        (3,'Widget','Electronics','South',25.0,5,125.0),
        (4,'Doohickey','Accessories','East',8.0,50,400.0),
        (5,'Gadget','Electronics','East',150.0,7,1050.0),
        (6,'Widget','Electronics','East',25.0,4,100.0),
        (7,'Doohickey','Accessories','North',8.0,20,160.0),
        (8,'Gadget','Electronics','North',150.0,2,300.0),
        (9,'Widget','Electronics','West',25.0,6,150.0),
        (10,'Doohickey','Accessories','South',8.0,15,120.0),
        (11,'Thingamajig','Accessories','North',200.0,1,200.0),
        (12,'Thingamajig','Accessories','East',200.0,4,800.0),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO orders VALUES (?,?,?,?,?,?,?)', rows
    )
    products = [
        ('Widget','Electronics',25.0),
        ('Gadget','Electronics',150.0),
        ('Doohickey','Accessories',8.0),
        ('Thingamajig','Accessories',200.0),
    ]
    cur.executemany(
        'INSERT OR IGNORE INTO products VALUES (?,?,?)', products
    )
    conn.commit()


def run_query(conn, sql, params=()):
    cur = conn.cursor()
    cur.execute(sql, params)
    cols = [col[0] for col in cur.description]
    return [dict(zip(cols, row)) for row in cur.fetchall()]


def filter_orders(conn, region=None, category=None, min_revenue=None):
    conditions = []
    params = []
    if region is not None:
        conditions.append('region = ?')
        params.append(region)
    if category is not None:
        conditions.append('category = ?')
        params.append(category)
    if min_revenue is not None:
        conditions.append('revenue >= ?')
        params.append(min_revenue)
    where = ('WHERE ' + ' AND '.join(conditions)) if conditions else ''
    sql = f'SELECT * FROM orders {where} ORDER BY order_id'
    return run_query(conn, sql, tuple(params))


def group_revenue(conn, group_col):
    sql = (
        f'SELECT {group_col}, SUM(revenue) AS total, '
        'COUNT(*) AS orders, ROUND(AVG(revenue), 2) AS avg_revenue '
        f'FROM orders GROUP BY {group_col} ORDER BY total DESC'
    )
    return run_query(conn, sql)


def join_summary(conn):
    sql = (
        'SELECT o.region, p.category, '
        'SUM(o.revenue) AS total_revenue, COUNT(*) AS order_count '
        'FROM orders o '
        'INNER JOIN products p ON o.product = p.product '
        'GROUP BY o.region, p.category '
        'ORDER BY total_revenue DESC'
    )
    return run_query(conn, sql)

## Step 1 — Create and Populate Database

In [ ]:
conn = sqlite3.connect(':memory:')
setup_db(conn)

n = conn.execute('SELECT COUNT(*) FROM orders').fetchone()[0]
assert n == 12
np = conn.execute('SELECT COUNT(*) FROM products').fetchone()[0]
assert np == 4
print(f'orders: {n} rows, products: {np} rows')

## Step 2 — run_query (SELECT + WHERE)

In [ ]:
all_orders = run_query(conn, 'SELECT * FROM orders')
assert len(all_orders) == 12
assert isinstance(all_orders[0], dict)
print(f'All orders: {len(all_orders)} rows')
print('Columns:', list(all_orders[0].keys()))

north = run_query(conn,
                   'SELECT * FROM orders WHERE region = ?', ('North',))
assert len(north) == 4
print(f'North orders: {len(north)}')

## Step 3 — filter_orders (dynamic WHERE)

In [ ]:
# No filter
assert len(filter_orders(conn)) == 12

# Single filter
acc = filter_orders(conn, category='Accessories')
assert len(acc) == 5
print(f'Accessories orders: {len(acc)}')

# Combined filter
east_elec = filter_orders(conn, region='East', category='Electronics')
assert len(east_elec) == 2
print(f'East+Electronics: {len(east_elec)} orders')

## Step 4 — group_revenue (GROUP BY + ORDER BY)

In [ ]:
by_product = group_revenue(conn, 'product')
assert len(by_product) == 4
assert by_product[0]['product'] == 'Gadget'
assert abs(by_product[0]['total'] - 1800.0) < 0.01
print('By product (top 2):')
for r in by_product[:2]:
    print(f"  {r['product']}: total={r['total']}, orders={r['orders']}")

by_region = group_revenue(conn, 'region')
assert len(by_region) == 4
assert abs(sum(r['total'] for r in by_region) - 4105.0) < 0.01
print(f'By region: {len(by_region)} groups, sum={sum(r["total"] for r in by_region)}')

## Step 5 — join_summary (INNER JOIN)

In [ ]:
result = join_summary(conn)
assert len(result) == 7
assert set(result[0].keys()) == {'region', 'category', 'total_revenue', 'order_count'}
assert abs(result[0]['total_revenue'] - 1200.0) < 0.01
print(f'Join summary: {len(result)} region/category combinations')
for r in result[:3]:
    print(f"  {r['region']} / {r['category']}: {r['total_revenue']}")

## Step 6 — Query into pandas

In [ ]:
q5_df = pd.read_sql_query('SELECT * FROM orders', conn)
assert q5_df.shape == (12, 7)
print(f'DataFrame shape: {q5_df.shape}')
print(q5_df[['product', 'region', 'revenue']].head(4).to_string(index=False))

q1 = filter_orders(conn, category='Accessories')
q1 = sorted(q1, key=lambda r: r['revenue'], reverse=True)
q2 = group_revenue(conn, 'region')
q3 = run_query(conn,
    'SELECT product, SUM(revenue) AS total FROM orders'
    ' GROUP BY product ORDER BY total DESC LIMIT 1')
q4 = join_summary(conn)

assert len(q1) == 5
assert len(q2) == 4
assert q3[0]['product'] == 'Gadget'
assert len(q4) == 7
print('All solution checks passed.')

conn.close()